# ★ Bonus · Where is everything, right now?

The catalog stores a **TLE** for every object — the orbital elements stamped at
an **epoch**. To draw *right now* you take each TLE, run it forward from its
epoch to the current instant (**propagation**, via **Orekit**'s SGP4/SDP4), and
keep the resulting position.

20,210 objects propagate in well under a second, so this is the whole catalog —
not a sample. Then we plot x / y / z in 3-D with gog4j.

In [0]:
addDependency("org.orekit:orekit:13.0.3");
addDependency("org.dflib:dflib:2.0.0-M7");
addDependency("org.dflib:dflib-parquet:2.0.0-M7");
addDependency("org.jtaccuino:gog4j:0.5-SNAPSHOT");
addDependency("org.jtaccuino:gog4j-hardwood:0.5-SNAPSHOT");

import org.dflib.DataFrame;
import org.dflib.parquet.Parquet;
import org.orekit.data.DataContext;
import org.orekit.time.AbsoluteDate;
import org.orekit.time.TimeScale;
import org.orekit.propagation.analytical.tle.TLE;
import org.orekit.propagation.analytical.tle.TLEPropagator;
import org.orekit.utils.PVCoordinates;
import org.jtaccuino.gog.*;
import org.jtaccuino.gog.labs.Labs;
import org.jtaccuino.gog.render.SvgExporter;
import org.jtaccuino.gog.theme.Theme;
import org.jtaccuino.gog.hardwood.HardwoodTable;
import java.time.Instant;
import java.util.ArrayList;
import java.util.LinkedHashMap;
import java.util.List;
import java.util.Map;
import java.nio.file.Files;
import java.nio.file.Path;
import static org.dflib.Exp.*;
import javafx.animation.KeyFrame;
import javafx.animation.Timeline;
import javafx.util.Duration;
import javafx.scene.layout.Pane;
import javafx.scene.paint.Color;
import javafx.scene.shape.Circle;

String num(long v) { return String.format(java.util.Locale.ROOT, "%,d", v); }
String dec(double v, int p) { return String.format(java.util.Locale.ROOT, "%." + p + "f", v); }

// The notebook may live in a sub-folder (exercises/, bonus/, ...), so walk up
// from the notebook's own folder to find the repository's data/ directory.
Path dataset() {
    for (Path p = cwd.toAbsolutePath(); p != null; p = p.getParent()) {
        Path candidate = p.resolve("data/celestrak_gp_catalog.parquet");
        if (Files.isRegularFile(candidate)) return candidate;
    }
    throw new IllegalStateException("data/celestrak_gp_catalog.parquet not found above " + cwd);
}

var df = Parquet.loader().load(dataset());
var tai = DataContext.getDefault().getTimeScales().getTAI();
var now = new AbsoluteDate(Instant.now(), tai);

void save(GgFigure figure, String name) {
    try {
        var path = cwd.resolve(name);
        new SvgExporter().size(1200, 900).batchPoints(true).write(figure, path);
        println("saved %s (%s bytes)", name, num(Files.size(path)));
    } catch (Exception e) {
        println("could not save %s: %s", name, e);
    }
}

## 1 · Propagate every object to now

Build an Orekit `TLE` from the two stored lines, propagate it to `now`, and keep
the position in km. Some objects will not propagate (they were decaying when the
elements were published) — catch and skip those.

In [0]:
// ── given ── the columns we need, and the lists we will fill
var l1 = df.getColumn("tle_line1");
var l2 = df.getColumn("tle_line2");
var names = df.getColumn("object_name");
var classes = df.getColumn("orbit_class");
var kinds = df.getColumn("satcat_object_type");

var xs = new ArrayList<Double>();
var ys = new ArrayList<Double>();
var zs = new ArrayList<Double>();
var plotNames = new ArrayList<String>();
var plotClasses = new ArrayList<String>();
var plotKinds = new ArrayList<String>();

In [0]:
// ── TODO 1 ────────────────────────────────────────────────────────────
// For each row with TLE lines, propagate to `now` and append the position
// (km) and its labels to the lists above. Skip objects that will not propagate.
//
// hint: build a TLE from its two lines, then ask a TLEPropagator to propagate it to
//       `now`; read the position x/y/z in km. Catch and skip the ones that fail.

In [0]:
// ── given ─────────────────────────────────────────────────────────────
var radii = new ArrayList<Double>();
for (int i = 0; i < xs.size(); i++) {
    radii.add(Math.sqrt(xs.get(i) * xs.get(i) + ys.get(i) * ys.get(i) + zs.get(i) * zs.get(i)));
}
double minR = radii.stream().mapToDouble(Double::doubleValue).min().orElse(0);
double maxR = radii.stream().mapToDouble(Double::doubleValue).max().orElse(0);
println("propagated %s of %s objects to the current instant", num(xs.size()), num(df.height()));
println("distance from Earth centre: %s km .. %s km", num((long) minR), num((long) maxR));

## 2 · The 3-D point cloud

Look at it from outside: x / y / z, coloured by orbit class. The shells **are**
the orbit families — a dense ball (LEO), a sparse shell (MEO), one thin ring
(GEO), and long streaky arcs (HEO).

In [0]:
// ── TODO 2 ────────────────────────────────────────────────────────────
// Build a HardwoodTable from x / y / z and the labels, then plot a 3-D
// scatter coloured by orbit_class into `p`.
//
// hint: turn the lists into a table, then start a 3-D plot — how do you map x, y
//       and z, and which point geom is the 3-D one?

In [0]:
// ── given ─────────────────────────────────────────────────────────────
display(p);
save(p, "orbits-now-3d.svg");

## 3 · A flat view: payloads and debris

Seen from above the pole, colour by `satcat_object_type`. The geostationary ring
is obvious, and the debris from the three big break-ups shows up as clumps.

In [0]:
// ── TODO 3 ────────────────────────────────────────────────────────────
// Same table, but look straight down: x vs y, coloured by satcat_object_type.
//
// hint: the same table, but only x and y — coloured by the object-type column.

## ★ Bonus — make it move

A static cloud is a picture; motion is a plot. Precompute each object's position
at a series of instants, then play the frames back as a flip-book.

We keep it to **LEO** and a few hundred objects so it stays smooth — that is the
same "focus on LEO" idea as limiting the axes, done as a **filter on the data**
rather than a change to the plot.

In [0]:
// ── ★ BONUS ───────────────────────────────────────────────────────────
// Precompute a trajectory: for each frame, where every object is.
var leo = df.rows($str("orbit_class").eq("LEO")).select();
var leoL1 = leo.getColumn("tle_line1");
var leoL2 = leo.getColumn("tle_line2");

int fleetSize = 300;           // objects in the animation
int frames = 48;               // frames per lap
double secondsPerFrame = 120;  // two minutes of real motion per frame

var fleetL1 = new ArrayList<String>();
var fleetL2 = new ArrayList<String>();
for (int i = 0; i < leo.height() && fleetL1.size() < fleetSize; i++) {
    Object a = leoL1.get(i), b = leoL2.get(i);
    if (a != null && b != null) {
        fleetL1.add(a.toString());
        fleetL2.add(b.toString());
    }
}

// trajectory[frame][object] = x, y, z in km
double[][][] trajectory = new double[frames][fleetL1.size()][3];
for (int f = 0; f < frames; f++) {
    var at = now.shiftedBy(f * secondsPerFrame);
    for (int i = 0; i < fleetL1.size(); i++) {
        try {
            var tle = new TLE(fleetL1.get(i), fleetL2.get(i), tai);
            var pos = TLEPropagator.selectExtrapolator(tle).propagate(at)
                        .getPVCoordinates().getPosition();
            trajectory[f][i][0] = pos.getX() / 1000.0;
            trajectory[f][i][1] = pos.getY() / 1000.0;
            trajectory[f][i][2] = pos.getZ() / 1000.0;
        } catch (Exception e) {
            // this object will not propagate - leave it at the origin
        }
    }
}
println("precomputed %d frames for %s LEO objects", frames, num(fleetL1.size()));

In [0]:
// ── ★ BONUS ───────────────────────────────────────────────────────────
// Play the frames: a plain JavaFX Pane, Earth in the middle, one dot per
// satellite. Every tick we project the current frame, move the dots, and slowly
// turn the view — the same kind of loop an animation needs.
double scale = 230.0 / 9000.0;      // km -> pixels (LEO reaches ~ 8400 km)
double cx = 240, cy = 240;
var view = new Pane();
view.setPrefSize(480, 480);
view.getChildren().add(new Circle(cx, cy, 6378.137 * scale, Color.web("#2e7d32")));

var dots = new ArrayList<Circle>();
for (int i = 0; i < fleetL1.size(); i++) {
    var dot = new Circle(0, 0, 1.5, Color.web("#ffb454"));
    dots.add(dot);
    view.getChildren().add(dot);
}

var azimuth = new double[]{0.0};
var frame = new int[]{0};
var timeline = new Timeline(new KeyFrame(Duration.millis(90), e -> {
    double cosA = Math.cos(azimuth[0]), sinA = Math.sin(azimuth[0]);
    int f = frame[0] % frames;
    for (int i = 0; i < dots.size(); i++) {
        double x = trajectory[f][i][0], y = trajectory[f][i][1], z = trajectory[f][i][2];
        double px = x * cosA - y * sinA;
        double pz = x * sinA + y * cosA;
        dots.get(i).setCenterX(cx + px * scale);
        dots.get(i).setCenterY(cy - (z * 0.75 + pz * 0.35) * scale);
    }
    azimuth[0] += 0.01;
    frame[0]++;
}));
timeline.setCycleCount(Timeline.INDEFINITE);
timeline.play();
display(view);
println("animating %s satellites — re-run the cell to start a fresh lap", num(dots.size()));

In [0]:
// ── check ─────────────────────────────────────────────────────────────
var pass = xs.size() >= 20000 && xs.size() <= 21207
        && xs.size() == ys.size() && xs.size() == zs.size();
if (pass) {
    println("✓ %s objects propagated to now · %s km .. %s km from Earth centre",
            num(xs.size()), num((long) minR), num((long) maxR));
} else {
    println("✗ propagated %s objects — check the TODOs above", num(xs.size()));
}

Every point is a real position, computed from a published element set at the
moment you ran the cell. Reload and run it again tomorrow and the cloud is
different.

Back to the [overview](../narrative/00-overview.md).